# Speed settings: how fast can the full match run without losing accuracy?

Runs the pipeline on the hand-checked HIL-HAZ half-1 window (frames 1066-3333, about 75 s of play)
with the current settings and with faster ones, each adding one change:

| run | change |
|---|---|
| current | baseline.env as used so far |
| fp16 | models in half precision |
| fp16_kp3 | + pitch keypoints every 3rd frame (the last pitch fit is reused in between) |
| fp16_kp3_noreid | + tracker without its small image model (players are now joined afterwards by our appearance model) |
| fp16_kp3_noreid_team3 | + team colours every 3rd frame |
| trt_kp3_noreid_team3 | the same with TensorRT engines (built once per GPU type, kept in Drive) |

Each run is scored on speed and accuracy: IDs against your hand-checked IDs, and players' pitch
positions and the ball against the current settings. Results: `MyDrive/Playbook/speed_settings/`.

**Videos to watch** (same folder, made at the end):
- `compare_current_vs_<fastest>.mp4`: current settings and the fastest run side by side, to check by
  eye that the faster settings track the same.
- `<fastest>_joined_players.mp4`: the fastest run with players joined by the appearance model
  (boxes in team colours: pink / cyan, goalkeepers yellow, referees magenta; orange dot = ball). This is
  how the full-match output will look. Note: the appearance model was trained on this match, so the
  joined IDs here look better than they will on a new match.

**Runtime:** L4 GPU, about 50-70 minutes. **Runtime -> Run all**; do not edit cells. If it disconnects,
Run all again: finished runs are skipped.

In [ ]:
# Cell 1 — GPU, Drive, inputs
import os, json, time, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
GPU = g.stdout.strip(); print('GPU:', GPU)
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
if not os.path.isdir(PBD):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
DEST = f'{PBD}/speed_settings'; os.makedirs(DEST, exist_ok=True)
VIDEO, LO, HI, FPS = f'{PBD}/benchmark/HIL-HAZ_half1.mp4', 1066, 3333, 30.0
NEEDED = [VIDEO, f'{PBD}/people_model/train/checkpoint_best_total.pth', f'{PBD}/ball_model/train/checkpoint_best_total.pth',
          f'{PBD}/field_model/roboflow_v10/weights.pt']
missing = [p for p in NEEDED if not os.path.exists(p)]
if missing:
    raise SystemExit(f'Missing in Drive: {missing}')
print('inputs found')

In [ ]:
# Cell 2 — Our repo and packages
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
!pip install -q "rfdetr[train]==1.11.0" "git+https://github.com/roboflow/sports.git@main" "pydantic-settings==2.4.0" "python-dotenv==1.0.1" "lap>=0.5.13,<0.6"
shutil.copy('baseline.env', '.env')
import pandas as pd, numpy as np
from tools.speed_eval import identity, agreement
GT = pd.read_csv('per_frame_tracks_half1_unified.csv', low_memory=False)
GT = GT[(GT.frame >= LO) & (GT.frame <= HI)]
print('ready')

In [ ]:
# Cell 3 — The runs (skips finished ones; TensorRT is installed only for the last one)
FAST = {'FAST_MODE': 'true', 'DETECT_EVERY_N': '1', 'TEAM_UPDATE_EVERY_N': '1', 'HOMOGRAPHY_EVERY_N': '3'}
RUNS = {
    'current': {},
    'fp16': {'MODEL_ACCEL': 'fp16'},
    'fp16_kp3': {'MODEL_ACCEL': 'fp16', **FAST},
    'fp16_kp3_noreid': {'MODEL_ACCEL': 'fp16', **FAST, 'BOTSORT_WITH_REID': 'false'},
    'fp16_kp3_noreid_team3': {'MODEL_ACCEL': 'fp16', **FAST, 'BOTSORT_WITH_REID': 'false', 'TEAM_UPDATE_EVERY_N': '3'},
    'trt_kp3_noreid_team3': {'MODEL_ACCEL': 'tensorrt', **FAST, 'BOTSORT_WITH_REID': 'false', 'TEAM_UPDATE_EVERY_N': '3'},
}
COLS = ['frame', 'raw_tracker_id', 'track_id', 'display_track_id', 'class_id', 'conf', 'x1', 'y1', 'x2', 'y2',
        'x_m', 'y_m', 'team_id', 'ball_interpolated']
RES = f'{DEST}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}
for name, env_over in RUNS.items():
    if name in results:
        print(name, 'already done'); continue
    if env_over.get('MODEL_ACCEL') == 'tensorrt':
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr[tensorrt]==1.11.0'], capture_output=True, text=True)
        print('TensorRT install:', 'ok' if r.returncode == 0 else r.stderr[-500:])
    local = f'/content/speed_{name}'
    shutil.rmtree(local, ignore_errors=True)
    env = {**os.environ, **env_over}
    t0 = time.time()
    r = subprocess.run([sys.executable, 'main.py', '--source-video', VIDEO, '--start-frame', str(LO), '--end-frame', str(HI),
                        '--out-dir', local, '--no-video', '--enable-team'], capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(name, 'FAILED:', r.stderr[-1500:])
        results[name] = {'error': r.stderr[-1500:], 'settings': env_over}
        json.dump(results, open(RES, 'w'), indent=1); continue
    kpi = json.load(open(f'{local}/kpi_summary.json'))
    t = pd.read_csv(f'{local}/per_frame_tracks.csv', low_memory=False)
    t = t[[c for c in COLS if c in t.columns]]
    t.to_csv(f'{DEST}/{name}_tracks.csv.gz', index=False, compression='gzip')
    res = {'settings': env_over, 'gpu': GPU, 'wall_minutes': round((time.time() - t0) / 60, 1),
           'fps': round(kpi.get('effective_fps', 0), 2),
           'ms_per_frame': {k.replace('time_ms_per_frame_', ''): round(v, 1) for k, v in kpi.items()
                            if k.startswith('time_ms_per_frame_') and v >= 1},
           'ball_detect_coverage': round(kpi.get('ball_detect_coverage', 0), 4),
           'homography_ok_rate': round(kpi.get('homography_ok_rate', 0), 4),
           'identity_vs_hand_ids': identity(t, GT, FPS)}
    if name != 'current' and os.path.exists(f'{DEST}/current_tracks.csv.gz'):
        res['vs_current'] = agreement(t, pd.read_csv(f'{DEST}/current_tracks.csv.gz'))
    results[name] = res
    json.dump(results, open(RES, 'w'), indent=1)
    print(f"{name}: {res['fps']} frames/s, IDs {res['identity_vs_hand_ids']}, vs current {res.get('vs_current')}")
print('runs done')

In [ ]:
# Cell 4 — Videos to watch (saved to the same folder)
from tools.render_run import render
ok = {n: r for n, r in results.items() if 'error' not in r}
fastest = max(ok, key=lambda n: ok[n]['fps'])
print('fastest run:', fastest, ok[fastest]['fps'], 'frames/s')
FRAMES = range(LO, HI + 1)
V1 = f'{DEST}/compare_current_vs_{fastest}.mp4'
if not os.path.exists(V1):
    render(VIDEO, {'current': f'{DEST}/current_tracks.csv.gz', fastest: f'{DEST}/{fastest}_tracks.csv.gz'},
           '/content/compare.mp4', width=960, frames=FRAMES)
    shutil.copy('/content/compare.mp4', V1)
print('video 1:', V1)
MODEL = f'{PBD}/reid_appearance/appearance_model.pt'
V2 = f'{DEST}/{fastest}_joined_players.mp4'
if not os.path.exists(MODEL):
    print('appearance model not found, video 2 skipped:', MODEL)
elif not os.path.exists(V2):
    from tools.join_pieces import join
    joined, info = join(pd.read_csv(f'{DEST}/{fastest}_tracks.csv.gz'), VIDEO, MODEL)
    print('joined:', info)
    joined.to_csv('/content/joined.csv', index=False)
    joined.to_csv(f'{DEST}/{fastest}_joined_tracks.csv.gz', index=False, compression='gzip')
    render(VIDEO, {f'{fastest}, joined players': '/content/joined.csv'}, '/content/joined.mp4', width=1280,
           frames=FRAMES, id_col='joined_id', team_colors=True)
    shutil.copy('/content/joined.mp4', V2)
print('video 2:', V2)
print(f'\nDone. Tell Claude the results are in {DEST}.')